# Why does `run_sweep` need both `axes` and `apply`?

`run_sweep` lets you run many trials varying one or more parameters.
You write two things:

- **`axes`** — *what* varies. A mapping from a label to a list of
  values. Labels are arbitrary; they show up in the result table.
- **`apply`** — *how* to inject each value into the scenario. A mapping
  from the same label to a function `(scenario, value) -> None`.

The split exists because the library can't guess what your label means.
`"num_agents": 30` is meaningful to you, not to the sweeper — you tell
it `set_agent_count("...", 30)`.

In [ ]:
import logging
from datetime import datetime

# Silence jupedsim-scenarios' INFO/DEBUG output. See the howto
# "How do I inspect a scenario?" for the full list of levels.
logging.getLogger("jupedsim_scenarios").setLevel(logging.WARNING)

print(f"Executed on {datetime.now().strftime('%d.%m.%Y, %H:%M')}")

## Two axes

`axes` with multiple entries produces the **Cartesian product** of all
value lists. Here that's 3 × 3 = 9 conditions × 5 seeds = 45 trials.

In [2]:
sweep = run_sweep(
    base,
    axes={
        "num_agents":    [30, 40, 50],
        "desired_speed": [1.0, 1.2, 1.4],
    },
    apply={
        "num_agents":    lambda s, n: s.set_agent_count(0, n),
        "desired_speed": lambda s, v: s.set_agent_params(0, desired_speed=v),
    },
    seeds=range(100, 105),
    workers=4,
)
sweep.to_dataframe().head()

Using fallback logic: No journeys definedUsing fallback logic: No journeys defined

Using fallback logic: No journeys defined
Using fallback logic: No journeys defined
Processing with parameters: {'number': 30, 'radius': 0.2, 'v0': 1.0, 'flow_start_time': 0, 'flow_end_time': 10, 'percentage': None, 'distribution_mode': 'by_number', 'use_flow_spawning': False, 'use_premovement': False, 'premovement_distribution': 'gamma', 'premovement_param_a': None, 'premovement_param_b': None, 'premovement_seed': None, 'radius_distribution': 'constant', 'v0_distribution': 'constant', 'desired_speed': 1.0}
Using default parameters: v0=1.0, radius=0.2, n_agents=30

Processing with parameters: {'number': 30, 'radius': 0.2, 'v0': 1.0, 'flow_start_time': 0, 'flow_end_time': 10, 'percentage': None, 'distribution_mode': 'by_number', 'use_flow_spawning': False, 'use_premovement': False, 'premovement_distribution': 'gamma', 'premovement_param_a': None, 'premovement_param_b': None, 'premovement_seed': None, 'ra

,num_agents,desired_speed,seed,trial_index,success,evacuation_time,total_agents,agents_evacuated,agents_remaining,sqlite_path
0,30,1.0,100,0,True,41.38,30,30,0,/tmp/tmp0fiyq5yw.sqlite
1,30,1.0,101,1,True,42.42,30,30,0,/tmp/tmpenffaj4v.sqlite
2,30,1.0,102,2,True,41.95,30,30,0,/tmp/tmpgp2hc3h2.sqlite
3,30,1.0,103,3,True,39.23,30,30,0,/tmp/tmpg1ri7dye.sqlite
4,30,1.0,104,4,False,300.00,30,18,12,/tmp/tmp_bvelect.sqlite


## Paired (zipped) conditions

To pair values instead of crossing them — e.g. only the three
combinations `(30, 1.0)`, `(40, 1.2)`, `(50, 1.4)` — collapse them into
a single composite axis:

```python
axes={"cond": [(30, 1.0), (40, 1.2), (50, 1.4)]},
apply={"cond": lambda s, c: (
    s.set_agent_count(0, c[0]),
    s.set_agent_params(0, desired_speed=c[1]),
)},
```